In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\dedha\AppData\Local\Temp\ipykernel_26240\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
e:\Downloads\Telegram Desktop\plant_disease\plantguard\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../pdfs")

Found 6 PDF files to process

Processing: AGS 322 -Diseases-of-Field-Crops-and-Their-Management.pdf
  ✓ Loaded 54 pages

Processing: Common-diseases-of-vegetable-crops-and-their-management_Aug.-25-2016.pdf
  ✓ Loaded 51 pages

Processing: Diseases-of-Field-Crops-and-Their-Management.pdf
  ✓ Loaded 198 pages

Processing: Diseases_Field-Crops_a_20.04.2020.pdf
  ✓ Loaded 119 pages

Processing: Plant Diseases and Disorders - NCSUpdf.pdf
  ✓ Loaded 76 pages

Processing: tom002.pdf
  ✓ Loaded 6 pages

Total documents loaded: 504


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® Word 2010', 'creator': 'Microsoft® Word 2010', 'creationdate': '2020-03-26T16:19:57+05:30', 'source': '..\\pdfs\\AGS 322 -Diseases-of-Field-Crops-and-Their-Management.pdf', 'file_path': '..\\pdfs\\AGS 322 -Diseases-of-Field-Crops-and-Their-Management.pdf', 'total_pages': 54, 'format': 'PDF 1.5', 'title': '', 'author': 'Dr Aditi', 'subject': '', 'keywords': '', 'moddate': '2020-03-26T16:19:57+05:30', 'trapped': '', 'modDate': "D:20200326161957+05'30'", 'creationDate': "D:20200326161957+05'30'", 'page': 0, 'source_file': 'AGS 322 -Diseases-of-Field-Crops-and-Their-Management.pdf', 'file_type': 'pdf'}, page_content='Diseases of Field Crops and Their Management \n1 \n \n \nAGS 322- Diseases of field and horticultural crops and their management \n1. Diseases of Wheat \n \n \nBlack or stem rust - Puccinia graminis tritici \nSymptoms \nSymptoms are produced on almost all aerial parts of the wheat plant but are most \ncommon on stem, leaf sheaths and

In [5]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [6]:
chunks=split_documents(all_pdf_documents)
chunks

Split 504 documents into 1024 chunks

Example chunk:
Content: Diseases of Field Crops and Their Management 
1 
 
 
AGS 322- Diseases of field and horticultural crops and their management 
1. Diseases of Wheat 
 
 
Black or stem rust - Puccinia graminis tritici 
...
Metadata: {'producer': 'Microsoft® Word 2010', 'creator': 'Microsoft® Word 2010', 'creationdate': '2020-03-26T16:19:57+05:30', 'source': '..\\pdfs\\AGS 322 -Diseases-of-Field-Crops-and-Their-Management.pdf', 'file_path': '..\\pdfs\\AGS 322 -Diseases-of-Field-Crops-and-Their-Management.pdf', 'total_pages': 54, 'format': 'PDF 1.5', 'title': '', 'author': 'Dr Aditi', 'subject': '', 'keywords': '', 'moddate': '2020-03-26T16:19:57+05:30', 'trapped': '', 'modDate': "D:20200326161957+05'30'", 'creationDate': "D:20200326161957+05'30'", 'page': 0, 'source_file': 'AGS 322 -Diseases-of-Field-Crops-and-Their-Management.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® Word 2010', 'creator': 'Microsoft® Word 2010', 'creationdate': '2020-03-26T16:19:57+05:30', 'source': '..\\pdfs\\AGS 322 -Diseases-of-Field-Crops-and-Their-Management.pdf', 'file_path': '..\\pdfs\\AGS 322 -Diseases-of-Field-Crops-and-Their-Management.pdf', 'total_pages': 54, 'format': 'PDF 1.5', 'title': '', 'author': 'Dr Aditi', 'subject': '', 'keywords': '', 'moddate': '2020-03-26T16:19:57+05:30', 'trapped': '', 'modDate': "D:20200326161957+05'30'", 'creationDate': "D:20200326161957+05'30'", 'page': 0, 'source_file': 'AGS 322 -Diseases-of-Field-Crops-and-Their-Management.pdf', 'file_type': 'pdf'}, page_content='Diseases of Field Crops and Their Management \n1 \n \n \nAGS 322- Diseases of field and horticultural crops and their management \n1. Diseases of Wheat \n \n \nBlack or stem rust - Puccinia graminis tritici \nSymptoms \nSymptoms are produced on almost all aerial parts of the wheat plant but are most \ncommon on stem, leaf sheaths and

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1977.68it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\dedha\AppData\Local\Temp\ipykernel_26240\2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [9]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create — don't delete!
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG",
                          "hnsw:space": "cosine"
                          },
                embedding_function=None
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 1024


In [10]:
import shutil
shutil.rmtree(r"E:\Downloads\Telegram Desktop\plant_disease\plantguard\data\vector_store", ignore_errors=True)

vectorstore = VectorStore()
texts = [doc.page_content for doc in chunks]
embeddings = embedding_manager.generate_embeddings(texts)
vectorstore.add_documents(chunks, embeddings)

print("Final count:", vectorstore.collection.count())

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 1024
Generating embeddings for 1024 texts...


Batches: 100%|██████████| 32/32 [00:40<00:00,  1.27s/it]


Generated embeddings with shape: (1024, 384)
Adding 1024 documents to vector store...
Successfully added 1024 documents to vector store
Total documents in collection: 2048
Final count: 2048


In [11]:
#retriever pipeline
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                # n_results=top_k
                n_results=min(top_k, self.vector_store.collection.count())
            )
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [13]:
rag_retriever


In [14]:
rag_retriever.retrieve("what is economic importance")

Retrieving documents for query: 'what is economic importance'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 45.06it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_34785c95_115',
  'content': 'Major attributes\n• Mostly fungi and oomycetes\n• Inoculum sources include:\n– debris of previous crop\n– contaminated seeds and irrigation waster\n• Spores can travel several miles aided by \nwind and/or rain\n• Excessive and prolonged moisture \nconditions may promote disease\n• Capable of causing significant crop loss \nunder favorable conditions',
  'metadata': {'source': '..\\pdfs\\Common-diseases-of-vegetable-crops-and-their-management_Aug.-25-2016.pdf',
   'producer': 'Microsoft® PowerPoint® 2016',
   'creationDate': "D:20160901160231-05'00'",
   'total_pages': 51,
   'format': 'PDF 1.5',
   'creator': 'Microsoft® PowerPoint® 2016',
   'file_path': '..\\pdfs\\Common-diseases-of-vegetable-crops-and-their-management_Aug.-25-2016.pdf',
   'author': 'naidu',
   'page': 27,
   'keywords': '',
   'title': 'Slide 1',
   'moddate': '2016-09-01T16:02:31-05:00',
   'file_type': 'pdf',
   'doc_index': 115,
   'creationdate': '2016-09-01T16:02:31-05

In [15]:
import os
from dotenv import load_dotenv
load_dotenv()



True

In [16]:
from groq import Groq
groq_api_key = os.environ.get("GROQ_API_KEY")

class RAGPipeline:
    def __init__(self, retriever: RAGRetriever, groq_api_key: str, model: str = "llama-3.1-8b-instant"):
        self.retriever = retriever
        self.client = Groq(api_key=groq_api_key)
        self.model = model

    def ask(self, query: str, top_k: int = 5) -> str:
        docs = self.retriever.retrieve(query, top_k=top_k)
        
        if not docs:
            return "I can only answer questions related to crop diseases."
        
        best_score = docs[0]['similarity_score']
        if best_score < 0.3:
            return "I can only answer questions related to crop diseases and agriculture. Please ask something relevant."
        
        context = "\n\n".join([d['content'] for d in docs])
        
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {
                    "role": "system",
                    "content": """You are an expert agricultural assistant specializing in crop diseases.
Answer the question directly and concisely based on the provided context.
If the context doesn't contain enough information, say so clearly instead of inferring.
Always mention the specific crop and disease name when available."""
                },
                {
                    "role": "user",
                    "content": f"Context:\n{context}\n\nQuestion: {query}"
                }
            ]
        )
        
        return response.choices[0].message.content

# Initialize
pipeline = RAGPipeline(
    retriever=rag_retriever,
    groq_api_key=os.environ.get("GROQ_API_KEY")
)

# Test
answer = pipeline.ask("what causes the holes in crop leaves")
print(answer)

Retrieving documents for query: 'what causes the holes in crop leaves'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 45.78it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


The passage does not specify what causes the holes in crop leaves directly. It provides a description of various diseases and their symptoms, but holes are not mentioned as a symptom of any of the listed diseases. 

However, it mentions that symptoms of Angular leaf spot, Bacterial blight, and downy mildew can cause leaf spotting, but does not specifically mention holes.


In [ ]:
import whisper
import sounddevice as sd
import numpy as np
import scipy.io.wavfile as wav

model = whisper.load_model("base")


def record_and_transcribe(duration=10, sample_rate=16000):
    print("🎤 Listening...")
    audio = sd.rec(int(duration * sample_rate),
                   samplerate=sample_rate, channels=1, dtype='float32')
    sd.wait()
    wav.write("temp.wav", sample_rate, audio)
    result = model.transcribe("temp.wav")  # auto-detects Hindi or English
    return result["text"]

# Test
query = record_and_transcribe()
print("You said:", query)
answer = pipeline.ask(query)
print(answer)

🎤 Listening...


e:\Downloads\Telegram Desktop\plant_disease\plantguard\.venv\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


FileNotFoundError: [WinError 2] The system cannot find the file specified